# Notebook 03 - Archive Backup Pack and Restore

This notebook demonstrates the latest `fintech-market-ingestion` archive backup workflow for notebook-first use in Colab.

Archive backup packs are intended for larger curated datasets, especially partitioned Parquet-style folders under `data/curated/`. They are useful because mounted Google Drive can be slow when copying many small files one by one.

This version keeps the same session discipline used by the earlier notebooks:

- initialize a local project session,
- read `SESSION_ID` from the session manifest,
- derive Drive paths from `{SESSION_ID}`,
- derive the archive/backup identifier from `{SESSION_ID}`,
- pack local curated data into Drive-backed archive shards,
- validate and inspect the archive before restore,
- restore into local runtime storage rather than analyzing directly from Drive.


## Tutorial context

This notebook is part of a notebook-first tutorial series:

```text
Notebook 00 - Setup and Storage Overview
Notebook 01 - Extraction Layer: Daily Bars Backfill
Notebook 02 - Session Save and Restore
Notebook 03 - Archive Backup Pack and Restore
Future notebook - Storage Strategy Comparison
```

The active local workspace is:

```text
/content/fintech-market-ingestion-demo
```

The mounted Google Drive persistence root is:

```text
/content/drive/MyDrive/{DRIVE_FOLDER_NAME}
```

The active archive pack will be stored under a session-scoped Drive directory:

```text
/content/drive/MyDrive/{DRIVE_FOLDER_NAME}/sessions/{SESSION_ID}/backups/{ARCHIVE_ID}
```

`SESSION_ID` should remain the stable bridge across setup, extraction, session save, archive backup, and restore notebooks.


## 1. Install required packages

Run this in a fresh notebook runtime.

The tutorial installs `pandas-market-calendars` explicitly and installs `fintech-market-ingestion` from TestPyPI.

In [ ]:
!python -m pip install "pandas-market-calendars>=5.0"
!python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion

## 2. Verify required CLI commands

This notebook primarily uses:

```text
fintech-init-project
fintech-backup-data
```

This archive notebook checks only the CLI commands it uses so the source stays aligned with the current backup-pack workflow without promoting older restore command shapes.


In [ ]:
import shutil

required_commands = [
    "fintech-init-project",
    "fintech-backup-data",
]

for command in required_commands:
    command_path = shutil.which(command)
    print(f"{command}: {command_path if command_path else 'NOT FOUND'}")


## 3. Authorize and mount Google Drive

Run this cell in Google Colab to authorize Google Drive access.

The package does not mount Drive itself. Drive access is user-initiated here, and later commands treat Drive as a normal mounted filesystem path.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Define local and Drive roots

The local workspace is the active runtime workspace.

The Google Drive root is persistence storage only. Archive packs are stored there so they can survive Colab runtime resets.

Keep `WORKSPACE_ROOT` and `DRIVE_PROJECT_ROOT` consistent with the earlier notebooks in this tutorial series.


In [ ]:
from pathlib import Path
import json

WORKSPACE_ROOT = Path("/content/fintech-market-ingestion-demo")
DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
DRIVE_SESSIONS_ROOT = DRIVE_PROJECT_ROOT / "sessions"
SOURCE_DATASET_ROOT = WORKSPACE_ROOT / "data" / "curated"

WORKSPACE_ROOT_STR = WORKSPACE_ROOT.as_posix()
SOURCE_DATASET_ROOT_STR = SOURCE_DATASET_ROOT.as_posix()

DRIVE_FOLDER_NAME_IS_PLACEHOLDER = DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME"
if DRIVE_FOLDER_NAME_IS_PLACEHOLDER:
    print("Update DRIVE_FOLDER_NAME before running Drive archive cells in Colab.")

DRIVE_PROJECT_ROOT_STR = DRIVE_PROJECT_ROOT.as_posix()
DRIVE_SESSIONS_ROOT_STR = DRIVE_SESSIONS_ROOT.as_posix()

print("Workspace root:", WORKSPACE_ROOT)
print("Source dataset root:", SOURCE_DATASET_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Drive sessions root:", DRIVE_SESSIONS_ROOT)

## 5. Initialize the project session

A standalone archive notebook should create or confirm a local project workspace before packing or restoring data.

This creates a session manifest under:

```text
artifacts/sessions/<session_id>/session_manifest.json
```

The session manifest is metadata only. It does not copy data, run ingestion, or perform a backup.

In [ ]:
!fintech-init-project \
  --root {WORKSPACE_ROOT_STR} \
  --notebooks \
  --with-session \
  --session-name archive_backup_demo


## 6. Extract `SESSION_ID` from the initialized notebook session

This cell reads the latest project-session manifest and extracts `SESSION_ID`.

Later shell commands use `{SESSION_ID}` directly through notebook variable interpolation. Do not replace this with a hardcoded folder name.


In [ ]:
session_manifest_paths = sorted(
    (WORKSPACE_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime,
)

if not session_manifest_paths:
    raise FileNotFoundError("No session_manifest.json files found. Run fintech-init-project first.")

SESSION_MANIFEST_PATH = session_manifest_paths[-1]

with SESSION_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    SESSION_MANIFEST = json.load(file)

SESSION_ID = SESSION_MANIFEST["session_id"]

print("Session manifest:", SESSION_MANIFEST_PATH)
print("SESSION_ID:", SESSION_ID)
print("Session name:", SESSION_MANIFEST.get("session_name"))


## 7. Prepare `{SESSION_ID}`-based Google Drive directories

This cell creates a Google Drive session directory only if it does not already exist.

The archive backup root is nested under the active `SESSION_ID`, so each notebook session can keep its backup packs separate.

The archive identifier is also derived from `SESSION_ID`:

```python
ARCHIVE_ID = f"curated-data-{SESSION_ID}"
```


In [ ]:
DRIVE_SESSION_ROOT = DRIVE_SESSIONS_ROOT / SESSION_ID
DRIVE_BACKUP_ROOT = DRIVE_SESSION_ROOT / "backups"

ARCHIVE_ID = f"curated-data-{SESSION_ID}"

if DRIVE_FOLDER_NAME_IS_PLACEHOLDER:
    raise RuntimeError("Update DRIVE_FOLDER_NAME before creating Drive archive directories.")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive in Colab before creating Drive archive directories.")

DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_SESSION_ROOT_STR = DRIVE_SESSION_ROOT.as_posix()
DRIVE_BACKUP_ROOT_STR = DRIVE_BACKUP_ROOT.as_posix()

print("SESSION_ID:", SESSION_ID)
print("ARCHIVE_ID:", ARCHIVE_ID)
print("Drive session root exists:", DRIVE_SESSION_ROOT.exists())
print("Drive session root:", DRIVE_SESSION_ROOT)
print("Drive backup root exists:", DRIVE_BACKUP_ROOT.exists())
print("Drive backup root:", DRIVE_BACKUP_ROOT)


## 8. Optional: restore from a previous Drive session

Most runs should use the current `SESSION_ID`.

If you intentionally want to restore a backup from a previous session, list available Drive session directories and set `RESTORE_SESSION_ID` to one of them. Leave it as `SESSION_ID` to use the current notebook session.

For a previous session, also set `RESTORE_ARCHIVE_ID` to the archive ID that was created in that session.


In [ ]:
available_drive_sessions = sorted(
    path.name for path in DRIVE_SESSIONS_ROOT.glob("*") if path.is_dir()
)

print("Available Drive session IDs:")
for session_id in available_drive_sessions:
    print(" -", session_id)

RESTORE_SESSION_ID = SESSION_ID
RESTORE_ARCHIVE_ID = ARCHIVE_ID

RESTORE_BACKUP_ROOT = DRIVE_SESSIONS_ROOT / RESTORE_SESSION_ID / "backups"
RESTORE_BACKUP_ROOT_STR = RESTORE_BACKUP_ROOT.as_posix()

print("Using RESTORE_SESSION_ID:", RESTORE_SESSION_ID)
print("Using RESTORE_ARCHIVE_ID:", RESTORE_ARCHIVE_ID)
print("Restore backup root exists:", RESTORE_BACKUP_ROOT.exists())
print("Restore backup root:", RESTORE_BACKUP_ROOT)


## 9. Confirm or create demo curated data

Archive backup packs need files under `data/curated/`.

If you already ran Notebook 01, this folder may contain real daily bars. If not, this cell creates a few tiny demo `.parquet`-named files so the backup commands can run in a standalone tutorial.

These demo files are placeholders only. They are not real market data.

In [ ]:
SOURCE_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

existing_parquet_files = list(SOURCE_DATASET_ROOT.rglob("*.parquet"))

if existing_parquet_files:
    print(f"Found {len(existing_parquet_files)} existing parquet files.")
else:
    demo_files = [
        SOURCE_DATASET_ROOT / "bars_daily" / "symbol=AAPL" / "year=2024" / "part-000.parquet",
        SOURCE_DATASET_ROOT / "bars_daily" / "symbol=MSFT" / "year=2024" / "part-000.parquet",
        SOURCE_DATASET_ROOT / "bars_daily" / "symbol=NVDA" / "year=2024" / "part-000.parquet",
    ]

    for index, path in enumerate(demo_files, start=1):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(f"demo parquet placeholder {index}\n".encode("utf-8"))

    print(f"Created {len(demo_files)} demo .parquet placeholder files.")

for path in sorted(SOURCE_DATASET_ROOT.rglob("*.parquet"))[:10]:
    print(path)

## 10. Dry-run the archive backup pack

Start with a dry run. This previews the backup pack plan without writing the pack directory, manifest, or shard files.

The archive ID includes the active notebook `SESSION_ID`:

```text
curated-data-{SESSION_ID}
```


In [ ]:
!fintech-backup-data pack \
  --workspace-root {WORKSPACE_ROOT_STR} \
  --source-dataset-root {SOURCE_DATASET_ROOT_STR} \
  --backup-root {DRIVE_BACKUP_ROOT_STR} \
  --backup-id {ARCHIVE_ID} \
  --shard-size-mb 512 \
  --dry-run


## 11. Create the archive backup pack

After reviewing the dry-run output, run this cell to create the backup pack.

The pack is written to the active session backup directory on Drive using the `{SESSION_ID}`-derived `ARCHIVE_ID`.


In [ ]:
!fintech-backup-data pack \
  --workspace-root {WORKSPACE_ROOT_STR} \
  --source-dataset-root {SOURCE_DATASET_ROOT_STR} \
  --backup-root {DRIVE_BACKUP_ROOT_STR} \
  --backup-id {ARCHIVE_ID} \
  --shard-size-mb 512


## 12. Confirm the backup pack exists

This cell checks that the archive pack directory was created for the current `SESSION_ID` and `ARCHIVE_ID`.


In [ ]:
BACKUP_PACK_DIR = DRIVE_BACKUP_ROOT / ARCHIVE_ID
BACKUP_PACK_DIR_STR = BACKUP_PACK_DIR.as_posix()

print("SESSION_ID:", SESSION_ID)
print("ARCHIVE_ID:", ARCHIVE_ID)
print("Backup pack directory exists:", BACKUP_PACK_DIR.exists())
print("Backup pack directory:", BACKUP_PACK_DIR)

if BACKUP_PACK_DIR.exists():
    for path in sorted(BACKUP_PACK_DIR.rglob("*"))[:20]:
        print(path)


## 13. Validate the backup pack

Validation checks that the backup pack is structurally safe to restore.

It verifies the manifest, shard files, checksums, and archive member paths before any restore is attempted.

In [ ]:
!fintech-backup-data validate \
  --backup-pack-dir {BACKUP_PACK_DIR_STR}

## 14. Inspect the backup pack

Inspection summarizes the backup pack without extracting it.

Use this to confirm included datasets, file counts, byte totals, and shard structure.

In [ ]:
!fintech-backup-data inspect \
  --backup-pack-dir {BACKUP_PACK_DIR_STR}

## 15. Choose a local restore target

For normal use, restore into:

```text
/content/fintech-market-ingestion-demo/data/curated
```

For a safer tutorial demonstration, this notebook restores into a separate check directory first.

The restore target is local Colab runtime storage. Avoid running analysis directly against the archive pack on Drive.


In [ ]:
RESTORE_ROOT = WORKSPACE_ROOT / "data" / "curated_restore_check"
RESTORE_ROOT.mkdir(parents=True, exist_ok=True)

RESTORE_ROOT_STR = RESTORE_ROOT.as_posix()

print("Restore root:", RESTORE_ROOT)
print("Restore root exists:", RESTORE_ROOT.exists())

## 16. Restore the backup pack

This command restores the archive backup pack into the local restore root.

The default-safe overwrite policy is `fail`, which stops before writing if existing target files would be replaced.

In [ ]:
!fintech-backup-data restore \
  --backup-pack-dir {BACKUP_PACK_DIR_STR} \
  --restore-root {RESTORE_ROOT_STR} \
  --overwrite-policy fail

## 17. Inspect restored files

After restore, confirm that files were written to the local restore root.

In a real workflow, analysis should run against local restored files, not directly against Google Drive.

In [ ]:
restored_files = sorted(RESTORE_ROOT.rglob("*.parquet"))

print("Restored parquet file count:", len(restored_files))
for path in restored_files[:20]:
    print(path)

## 18. Restore from a previous session backup

Use this section only when restoring an archive pack created by a previous notebook session.

Set `RESTORE_SESSION_ID` and `RESTORE_ARCHIVE_ID` in the earlier cell to the Drive session and archive pack you want.


In [ ]:
PREVIOUS_BACKUP_PACK_DIR = RESTORE_BACKUP_ROOT / RESTORE_ARCHIVE_ID
PREVIOUS_BACKUP_PACK_DIR_STR = PREVIOUS_BACKUP_PACK_DIR.as_posix()

print("Previous/session restore backup pack:", PREVIOUS_BACKUP_PACK_DIR)
print("Exists:", PREVIOUS_BACKUP_PACK_DIR.exists())


In [ ]:
# Uncomment to validate a backup from RESTORE_SESSION_ID / RESTORE_ARCHIVE_ID.
# !fintech-backup-data validate \
#   --backup-pack-dir {PREVIOUS_BACKUP_PACK_DIR_STR}

# Uncomment to inspect a backup from RESTORE_SESSION_ID / RESTORE_ARCHIVE_ID.
# !fintech-backup-data inspect \
#   --backup-pack-dir {PREVIOUS_BACKUP_PACK_DIR_STR}

# Uncomment to restore a backup from RESTORE_SESSION_ID / RESTORE_ARCHIVE_ID into local curated data.
# !fintech-backup-data restore \
#   --backup-pack-dir {PREVIOUS_BACKUP_PACK_DIR_STR} \
#   --restore-root {SOURCE_DATASET_ROOT_STR} \
#   --overwrite-policy fail


## 19. Storage boundary

Archive backup packs are derived transfer artifacts.

They are useful for persistence and restore, but they are not the active dataset, not a database, and not a second source of truth.

The intended pattern is:

```text
initialize notebook session
        v
derive SESSION_ID from session_manifest.json
        v
work locally in /content
        v
pack local curated data into archive shards
        v
store backup pack on mounted Drive under sessions/{SESSION_ID}/backups/{ARCHIVE_ID}
        v
validate and inspect before restore
        v
restore back into local runtime storage
        v
run analysis from local restored files
```

Keep this distinction clear:

- `SESSION_ID` organizes notebook session state and Drive persistence.
- `ARCHIVE_ID` identifies a specific backup pack for that session.
- archive packs are disposable restore/transfer artifacts.
- local curated data remains the active analysis source.


## Notebook summary

In this notebook, you learned how to:

- install standalone runtime dependencies
- install `fintech-market-ingestion` from TestPyPI
- authorize and mount Google Drive
- initialize a local project session
- extract `SESSION_ID` from the session manifest
- create `{SESSION_ID}`-based Drive backup directories
- create a `{SESSION_ID}`-derived `ARCHIVE_ID`
- create demo curated files when no curated dataset exists
- dry-run an archive backup pack
- create a backup pack using `curated-data-{SESSION_ID}`
- validate and inspect the backup pack
- restore the backup pack into local runtime storage
- optionally target a previous Drive session and archive ID for restore

Rule of thumb:

> Use session save for workflow state.  
> Use archive backup packs for larger curated datasets.  
> Use `{SESSION_ID}` as the stable notebook-session bridge across setup, extraction, save, archive, and restore workflows.
